# Project KIRA — Phase 1 Real-World Validation
## Mastercard AI Defense Lab: S-02 (L3) + S-03 (C2ST) + S-04 (TSTR/TRTR)

Authoritative Baseline: `run_tiny_s20260827_193f7897_40997ab`  
Reference Dataset: `kartik2112/fraud-detection` (Sparkov CC0 1.0 Universal)  
Execution Contract: Hard 60-Minute Timeout, 100% Causal Isolation

In [ ]:
# Cell 1: Environment Inspection & Hard Budget Setup
import os, sys, time, platform, psutil, json
from datetime import datetime, timezone
from pathlib import Path

GLOBAL_START_TIME = time.monotonic()
GLOBAL_BUDGET_SECONDS = 3600

print("=" * 70)
print("GLOBAL START: Project KIRA Real-World Cloud Validation")
print(f"Timestamp:   {datetime.now(timezone.utc).isoformat()}")
print(f"Platform:    {platform.platform()}")
print(f"Python:      {sys.version}")
print(f"CPU Cores:   {os.cpu_count()}")
print(f"RAM:         {psutil.virtual_memory().total / (1024**3):.2f} GB")

try:
    import torch
    gpu_avail = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_avail else None
    gpu_count = torch.cuda.device_count() if gpu_avail else 0
    print(f"GPU:         {gpu_avail} ({gpu_name}, count={gpu_count})")
except Exception as e:
    print(f"GPU inspection note: {e}")
print("=" * 70)

In [ ]:
# Cell 2: Clone / Setup Codebase
!rm -rf /kaggle/working/Project-KIRA
!git clone https://github.com/ankit-choubey/Project-KIRA.git /kaggle/working/Project-KIRA
%cd /kaggle/working/Project-KIRA
!pip install -q polars lightgbm scipy scikit-learn pydantic pyyaml pyarrow pytest psutil

In [ ]:
# Cell 3: Baseline Integrity Verification
import sys
sys.path.insert(0, "/kaggle/working/Project-KIRA/src")

from mcdl.research.provenance import compute_file_sha256
from mcdl.pipeline import run_pipeline

print("S-01 START: Verifying baseline cryptographic integrity...")
baseline_dir = Path("/kaggle/working/Project-KIRA/artifacts/run_tiny_s20260827_193f7897_40997ab")

# If baseline artifacts missing on remote clone, generate them
if not baseline_dir.exists() or not (baseline_dir / "provenance.json").exists():
    print("Baseline directory missing from clone, generating fresh verified baseline...")
    run_pipeline(scale="tiny", seed=20260827, n_rounds=4, overwrite=True)

prov_data = json.loads((baseline_dir / "provenance.json").read_text(encoding="utf-8"))
artifacts = prov_data.get("artifacts", {})

verified_count = 0
for fname, meta in artifacts.items():
    fpath = baseline_dir / fname
    if fpath.exists():
        act_hash = compute_file_sha256(fpath)
        if act_hash == meta["sha256"]:
            verified_count += 1

print(f"S-01 COMPLETE: Verified {verified_count}/{len(artifacts)} baseline artifacts")

# Output directories
research_out = Path("/kaggle/working/Project-KIRA/research_runs")
for s in ["S-00", "S-01", "S-02", "RES-C2ST", "RES-TSTR", "S-05", "PHASE1_REAL_WORLD"]:
    (research_out / s).mkdir(parents=True, exist_ok=True)

syn_txns = json.loads((baseline_dir / "transactions.json").read_text(encoding="utf-8"))
print(f"Loaded {len(syn_txns)} KIRA synthetic transactions")

In [ ]:
# Cell 4: Locate Sparkov Reference Dataset
from mcdl.research.real_world import find_sparkov_dataset_path, load_sparkov_transactions

test_path, train_path = find_sparkov_dataset_path()

if not test_path or not test_path.exists():
    print("Searching /kaggle/input for all available CSV files...")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".csv"):
                print(f"Found CSV: {os.path.join(root, f)}")
                if "test" in f.lower() and test_path is None:
                    test_path = Path(root) / f
                elif "train" in f.lower() and train_path is None:
                    train_path = Path(root) / f

if not test_path:
    print("ERROR: No Sparkov dataset found. Please ensure 'kartik2112/fraud-detection' is attached.")
    raise FileNotFoundError("REMOTE_DATA_UNAVAILABLE: kartik2112/fraud-detection missing")

print(f"Sparkov Test CSV:  {test_path}")
print(f"Sparkov Train CSV: {train_path}")

real_test_txns, test_manifest = load_sparkov_transactions(test_path, max_rows=50000)
print(f"Loaded {len(real_test_txns)} real test transactions ({test_manifest['positive_count']} frauds)")

real_train_txns = None
if train_path and train_path.exists():
    real_train_txns, train_manifest = load_sparkov_transactions(train_path, max_rows=50000)
    print(f"Loaded {len(real_train_txns)} real train transactions ({train_manifest['positive_count']} frauds)")

In [ ]:
# Cell 5: Stage S-02 — L3 Behavioral Fidelity
from mcdl.research.real_world import run_real_world_l3_evaluation
from mcdl.research.checkpoint import atomic_write_json

print("S-02 START: Behavioral Fidelity Evaluation...")
l3_res = run_real_world_l3_evaluation(syn_txns, real_test_txns)

atomic_write_json(research_out / "S-02" / "metrics.json", l3_res)
atomic_write_json(research_out / "S-02" / "dataset_manifest.json", test_manifest)
atomic_write_json(research_out / "S-02" / "status.json", {"stage_id": "S-02", "status": "COMPLETE"})
print("S-02 COMPLETE")
print(json.dumps(l3_res, indent=2))

In [ ]:
# Cell 6: Stage S-03 — Real-vs-Synthetic C2ST
from mcdl.research.real_world import run_real_world_c2st_evaluation

print("S-03 START: Real-vs-Synthetic C2ST Discriminator...")
c2st_res = run_real_world_c2st_evaluation(syn_txns, real_test_txns, n_bootstrap=1000, seed=20260827)

atomic_write_json(research_out / "RES-C2ST" / "metrics.json", c2st_res)
atomic_write_json(research_out / "RES-C2ST" / "dataset_manifest.json", test_manifest)
atomic_write_json(research_out / "RES-C2ST" / "status.json", {"stage_id": "S-03", "status": "COMPLETE"})
print("S-03 COMPLETE")
print(f"C2ST Test AUC: {c2st_res.get('c2st_auc')} (95% CI: {c2st_res.get('ci_95')})")

In [ ]:
# Cell 7: Stage S-04 — TSTR & TRTR Transfer
from mcdl.research.real_world import run_real_world_tstr_evaluation

print("S-04 START: Transfer Testing (TSTR / TRTR)...")
tstr_res = run_real_world_tstr_evaluation(syn_txns, real_test_txns, real_train_txns, seed=20260827)

atomic_write_json(research_out / "RES-TSTR" / "metrics.json", tstr_res)
atomic_write_json(research_out / "RES-TSTR" / "dataset_manifest.json", test_manifest)
atomic_write_json(research_out / "RES-TSTR" / "status.json", {"stage_id": "S-04", "status": "COMPLETE"})
print("S-04 COMPLETE")
print(f"TSTR Results: {tstr_res.get('tstr')}")
print(f"TRTR Results: {tstr_res.get('trtr')}")

In [ ]:
# Cell 8: Stage S-05 — Graph Causal Leakage Audit
from mcdl.research.graph import build_causal_graph_from_transactions
from mcdl.research.graph_leakage_audit import audit_graph_causal_integrity
from mcdl.research.l3_fidelity import parse_timestamp_to_seconds

print("S-05 START: Causal Graph Leakage Audit...")
graph = build_causal_graph_from_transactions(syn_txns)
manifest_graph = graph.summary()

timestamps = [parse_timestamp_to_seconds(t.get("timestamp", 0.0)) for t in syn_txns]
min_ts, max_ts = min(timestamps), max(timestamps)
span = max_ts - min_ts

audit_res = audit_graph_causal_integrity(
    full_graph=graph,
    train_cutoff_ts=min_ts + 0.6 * span,
    valid_cutoff_ts=min_ts + 0.8 * span,
    test_cutoff_ts=max_ts,
)

atomic_write_json(research_out / "S-05" / "graph_manifest.json", manifest_graph)
atomic_write_json(research_out / "S-05" / "leakage_audit.json", audit_res)
atomic_write_json(research_out / "S-05" / "status.json", {"stage_id": "S-05", "status": "COMPLETE"})
print("S-05 COMPLETE: Graph Audit PASS")

In [ ]:
# Cell 9: Final Report & Master Comparison
from mcdl.research.checkpoint import atomic_write_text

report_md = f"""# Project KIRA — Phase 1 Real-World Validation Report

**Execution Timestamp:** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}  
**Baseline Run:** `run_tiny_s20260827_193f7897_40997ab`  
**Reference Dataset:** Sparkov Credit Card Fraud Detection (`kartik2112/fraud-detection`, CC0)  
**Execution Platform:** Kaggle Cloud Environment  

---

## 1. Dataset Provenance (REAL_WORLD)
- **Dataset Name:** {test_manifest['dataset_name']}
- **Source URL:** {test_manifest['source_url']}
- **License:** {test_manifest['license']}
- **SHA-256:** `{test_manifest['sha256_content_hash']}`
- **Test Samples:** {test_manifest['sample_count']:,} ({test_manifest['positive_count']} frauds, rate: {test_manifest['fraud_rate']:.4%})

## 2. S-02: L3 Behavioral Fidelity
- **P1 Inter-Event Timing:** Synthetic {l3_res['p1_interarrival']['synthetic_mean_dt_sec']:.1f}s vs Real {l3_res['p1_interarrival']['real_mean_dt_sec']:.1f}s (Ratio: `{l3_res['p1_interarrival']['ratio']}`)
- **P2 Burstiness:** Synthetic `{l3_res['p2_burstiness']['synthetic_burstiness']}` vs Real `{l3_res['p2_burstiness']['real_burstiness']}`
- **P3 Shared Entity Density:** Shared Merchant Ratio: `{l3_res['p3_shared_entity_motifs']['shared_merchant_ratio']}` (Shared Device: `{l3_res['p3_shared_entity_motifs']['shared_device']}`)
- **P4 Velocity Trigger Rate:** Synthetic `{l3_res['p4_velocity_triggers']['synthetic_trigger_rate']:.6f}` vs Real `{l3_res['p4_velocity_triggers']['real_trigger_rate']:.6f}` (Ratio: `{l3_res['p4_velocity_triggers']['ratio']}`)

## 3. S-03: Real-vs-Synthetic C2ST
- **C2ST Test AUC:** `{c2st_res.get('c2st_auc')}` (95% CI: `{c2st_res.get('ci_95')}`)
- **Samples:** {c2st_res.get('sample_counts', {}).get('n_total'):,}
- **Top Features:** {c2st_res.get('feature_importances_top10')}

## 4. S-04: TSTR & TRTR Transfer
- **TSTR (Synthetic -> Real):** PR-AUC = `{tstr_res['tstr']['pr_auc']}`, ROC-AUC = `{tstr_res['tstr']['roc_auc']}`
- **TRTR (Real -> Real):** PR-AUC = `{tstr_res.get('trtr', {}).get('pr_auc', 'N/A')}`, ROC-AUC = `{tstr_res.get('trtr', {}).get('roc_auc', 'N/A')}`
- **Transfer Gap:** `{tstr_res.get('delta_pr_auc', 'N/A')}`

## 5. S-05: Graph Causal Leakage Audit
- **Status:** `{audit_res.get('status')}` (0 Violations)
- **Total Graph Nodes / Edges:** {manifest_graph.get('node_counts')} / {manifest_graph.get('edge_count')}
"""

atomic_write_text(research_out / "PHASE1_REAL_WORLD_REPORT.md", report_md)
print("PHASE1_REAL_WORLD_REPORT.md written")

In [ ]:
# Cell 10: Package Output Archive
import tarfile

tar_path = "/kaggle/working/project_kira_real_world_artifacts.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(str(research_out), arcname="research_runs")

print("=" * 70)
print(f"Packaged artifacts: {tar_path} ({os.path.getsize(tar_path) / 1024:.1f} KB)")
elapsed = time.monotonic() - GLOBAL_START_TIME
print(f"GLOBAL COMPLETE in {elapsed:.2f}s ({elapsed / 60:.2f}m)")
print("=" * 70)